In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        # Web search requires higher token budget
        "max_tokens": 4096,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
messages = []

web_search_schema = {
    "type": "web_search_20260318",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)

response = chat(messages, tools=[web_search_schema])

In [9]:
def parse_message(message):
    """Pretty-print a Message's content blocks in a human-readable way.

    Deliberately omits `encrypted_content` on web search results — it's an
    opaque, tamper-evident blob (citation integrity + multi-turn replay),
    not something meant to be read or parsed.
    """
    for block in message.content:
        if block.type == "text":
            print("\n" + "=" * 60)
            print("ANSWER")
            print("=" * 60)
            print(block.text)

        elif block.type == "thinking":
            preview = block.thinking[:300] if block.thinking else "(omitted)"
            print(f"[thinking] {preview}")

        elif block.type == "server_tool_use":
            print(f"[tool call: {block.name}] {block.input}")

        elif block.type == "web_search_tool_result":
            content = block.content
            if isinstance(content, list):
                print(f"[web_search results: {len(content)}]")
                for r in content:
                    age = f"  ({r.page_age})" if getattr(r, "page_age", None) else ""
                    print(f"  - {r.title}{age}\n    {r.url}")
            else:
                # error case: WebSearchToolResultError
                print(f"[web_search error] {content.error_code}")

        elif block.type == "bash_code_execution_tool_result":
            result = block.content
            if getattr(result, "type", None) == "bash_code_execution_result":
                if result.stdout:
                    print(f"[stdout]\n{result.stdout}")
                if result.stderr:
                    print(f"[stderr]\n{result.stderr}")
            else:
                print(f"[code execution error] {result}")

        elif block.type == "text_editor_code_execution_tool_result":
            print(f"[file op result] {block.content}")

        else:
            print(f"[{block.type}] {block}")

    if message.stop_reason == "max_tokens":
        print(
            "\n⚠️  Response was truncated (stop_reason=max_tokens) — "
            "the answer above is incomplete. Raise max_tokens in chat()."
        )

In [10]:
parse_message(response)

[tool call: code_execution] {'code': '\nresults = await web_search({"query": "best exercises for gaining leg muscle"})\nimport json\nfor i, r in enumerate(results):\n    print(f"Result {i}: {r[\'title\']}")\n    print(r[\'content\'][:300])\n    print("---")\n'}
[tool call: web_search] {'query': 'best exercises for gaining leg muscle'}
[web_search results: 10]
  - Maximizing Muscle Hypertrophy: A Systematic Review of Advanced Resistance Training Techniques and Methods - PMC
    https://pmc.ncbi.nlm.nih.gov/articles/PMC6950543/
  - The impact of resistance training on gluteus maximus hypertrophy: a systematic review and meta-analysis - PMC
    https://pmc.ncbi.nlm.nih.gov/articles/PMC12018462/
  - Gluteus Maximus Activation during Common Strength and Hypertrophy Exercises: A Systematic Review - PMC
    https://pmc.ncbi.nlm.nih.gov/articles/PMC7039033/
  - Muscle strength gains per week are higher in the lower-body than the upper-body in resistance training experienced healthy young women